# Big Data y MLOps - Sesion 2
## Procesamiento distribuido: limpieza, transformacion y agregaciones con PySpark

En la sesion anterior cargaste el dataset de transacciones bancarias con PySpark y lo exploraste de forma basica. Aca vamos a trabajar la parte central del procesamiento distribuido:

1. Definir el schema de forma manual (en lugar de dejar que Spark lo infiera).
2. Limpiar nulos, duplicados y valores invalidos.
3. Crear columnas calculadas.
4. Aplicar filtros avanzados.
5. Calcular metricas agregadas con `groupBy`.

Al final de esta sesion vas a tener un DataFrame limpio y enriquecido, listo para persistir en MySQL.

## Recuperando la sesion de Spark y los datos

Cada notebook corre en un kernel distinto, asi que necesitamos volver a crear la `SparkSession` y volver a leer el archivo CSV que generamos en la sesion anterior.

In [ ]:
from pyspark.sql import SparkSession
import os

# Creamos una sesión de Spark para ejecutar nuestro procesamiento.
spark = SparkSession.builder \
    .appName("PY4_BigData_ProcesamientoDistribuido") \
    .getOrCreate()

os.makedirs("data", exist_ok=True)
ruta_csv = "data/transacciones_masivas.csv" # Guardamos la ruta del archivo CSV que vamos a procesar.


## Definiendo el schema manualmente

En la sesion anterior usamos `inferSchema=True`, que es comodo pero tiene un costo: Spark tiene que leer el archivo una vez extra solo para deducir los tipos de datos. En archivos muy grandes esto puede ser lento e incluso llegar a inferir un tipo incorrecto.

La alternativa es definir el schema nosotros mismos con `StructType` y `StructField`. Esto es mas rapido y mas seguro, porque nosotros controlamos exactamente que tipo de dato tiene cada columna.

In [ ]:
from pyspark.sql.types import (  # Importamos los tipos de datos que usaremos para definir el esquema.

    StructType,      # Permite crear la estructura completa del esquema.
    StructField,     # Permite definir cada columna del DataFrame.
    IntegerType,     # Tipo de dato para números enteros.
    StringType,      # Tipo de dato para texto.
    DoubleType,      # Tipo de dato para números decimales.
    TimestampType    # Tipo de dato para fechas y horas.
)

schema_transacciones = StructType([  # Creamos el esquema que tendrá nuestro DataFrame.

    StructField("id_transaccion", IntegerType(), True),  # ID de la transacción como entero y puede ser nulo.

    StructField("cliente_id", IntegerType(), True),  # ID del cliente como entero y puede ser nulo.

    StructField("categoria", StringType(), True),  # Categoría de la transacción como texto y puede ser nula.

    StructField("monto", DoubleType(), True),  # Monto de la transacción como número decimal y puede ser nulo.

    StructField("canal", StringType(), True),  # Canal de la transacción como texto y puede ser nulo.

    StructField("fecha_hora", TimestampType(), True),  # Fecha y hora de la transacción y puede ser nula.

])  # Cerramos la definición del esquema.



df_spark = spark.read.csv(  # Leemos el archivo CSV utilizando Spark.
    ruta_csv,  # Indicamos la ruta donde se encuentra el archivo.
    header=True,  # Indicamos que la primera fila contiene los nombres de las columnas.
    schema=schema_transacciones  # Indicamos el esquema que queremos utilizar.
)

df_spark.printSchema()  # Mostramos la estructura y los tipos de datos del DataFrame.

root
 |-- id_transaccion: integer (nullable = true)
 |-- cliente_id: integer (nullable = true)
 |-- categoria: string (nullable = true)
 |-- monto: double (nullable = true)
 |-- canal: string (nullable = true)
 |-- fecha_hora: timestamp (nullable = true)



Definir el esquema manualmente puede parecer tedioso, pero tiene varias ventajas:

- Eficiencia: Spark no tiene que analizar los datos para intentar descubrir qué tipo tiene cada columna.
- Control: nosotros indicamos exactamente qué tipo de dato debe tener cada columna.
- Menos errores: evitamos que Spark interprete incorrectamente los tipos de datos.
- Importante en Big Data: con archivos de cientos de GB o TB, evitar pasos innecesarios puede ahorrar tiempo y recursos.

## Limpieza: duplicados, nulos y valores invalidos

Recuerda que este dataset tiene, a proposito:

- Algunos `id_transaccion` duplicados.
- Valores nulos en `monto` y `categoria`.
- Algunos montos negativos, que no tienen sentido para una transaccion de compra.

Vamos a resolver estos tres problemas paso a paso, para que quede claro que hace cada operacion.

In [ ]:
from pyspark.sql import functions as F # Importamos las funciones de PySpark y les ponemos el alias F para poder usarlas de forma más corta

# Eliminamos duplicados exactos por id_transaccion
df_sin_duplicados = df_spark.dropDuplicates(["id_transaccion"])
#                  ↑
#            por el momento es solamente un plan! (solo usando .show() podemos ver un resulto)

# Eliminamos filas donde monto o categoria son nulos
df_sin_nulos = df_sin_duplicados.dropna(subset=["monto", "categoria"])
#                  ↑
#            por el momento es solamente un plan!

# Eliminamos montos invalidos (negativos)
df_limpio = df_sin_nulos.filter(F.col("monto") > 0)
#                  ↑
#            por el momento es solamente un plan!



# ACTION → ahora Spark executes it:

print("Filas originales:", df_spark.count())
print("Filas despues de limpiar:", df_limpio.count())


+--------------+----------+-----------+-------+-------------+-------------------+
|id_transaccion|cliente_id|  categoria|  monto|        canal|         fecha_hora|
+--------------+----------+-----------+-------+-------------+-------------------+
|             1|      2844| Tecnologia|  39.14|          Web|2024-02-29 22:39:00|
|             3|      2200|       ropa| 247.17|          Web|2024-01-21 05:02:00|
|             5|      2543|       ropa|  69.54|          Web|2024-02-04 18:26:00|
|             6|      2190|  Alimentos|  12.65|          App|2024-01-18 10:13:00|
|             9|      4810| TECNOLOGIA|  100.3|          App|2024-02-23 19:23:00|
|            12|      1687|ELECTRONICA| 848.59|          App|2024-01-25 22:38:00|
|            13|      4445|   Deportes|1025.34|          App|2024-01-03 08:59:00|
|            15|      1105|ELECTRONICA|  83.71|          App|2024-01-01 11:43:00|
|            16|      2430|       ROPA|  36.58|          Web|2024-01-19 14:12:00|
|            17|

### Parece lo mismo a Pandas?  

Muchas operaciones parecen muy similares:

| Pandas | PySpark |
|---|---|
| `df.drop_duplicates()` | `df.dropDuplicates()` |
| `df.dropna()` | `df.dropna()` |
| `df[df["monto"] > 0]` | `df.filter(F.col("monto") > 0)` |
<br>

**Una diferencia muy importante:**


```
print("Filas originales:", df_spark.count())
```
Spark es lazy, el elaboró ​​un plan. Su plan incluye:
```
dropDuplicates()
dropna()
filter()
 ```
 Y solo despues:
 ```
 count()
 ```
Eso es la acción que le indica a Spark:

"Ahora ejecuta la tarea y cuenta las filas resultantes".


## Estandarizando texto inconsistente

La columna `categoria` tiene el mismo valor escrito de formas distintas, por ejemplo `"Electronica"`, `"electronica"` y `"ELECTRONICA"`. Para Spark, estos tres valores son categorias diferentes, lo cual arruina cualquier `groupBy` que hagamos mas adelante.

Vamos a usar `F.initcap()` junto con `F.lower()` para dejar todo en un formato consistente (primera letra en mayuscula, el resto en minuscula).

In [ ]:
df_limpio = df_limpio.withColumn(  # Creamos/reemplazamos la columna "categoria".
    "categoria",  # Indicamos la columna que queremos modificar.
    F.initcap(  # Ponemos la primera letra de cada palabra en mayúscula.
        F.lower(  # Primero convertimos todo el texto a minúsculas.
            F.col("categoria")  # Seleccionamos la columna "categoria".
        )
    )
)

df_limpio.select("categoria").distinct().show()  # Mostramos las categorías únicas después de limpiarlas.

+-----------+
|  categoria|
+-----------+
|   Deportes|
|       Ropa|
|Electronica|
|  Alimentos|
| Tecnologia|
|      Hogar|
+-----------+



El equivalente en Pandas seria:
```
df_limpio["categoria"].drop_duplicates().to_frame().reset_index(drop=True)
```

| PySpark              | Pandas                        |
| -------------------- | ----------------------------- |
| `F.col("categoria")` | `df["categoria"]`             |
| `F.lower()`          | `.str.lower()`                |
| `F.initcap()`        | `.str.title()`                |
| `.distinct()`        | `.drop_duplicates()`          |
| `.show()`            | `print()`                     |


## Columnas calculadas

Ahora que los datos estan limpios, vamos a agregar una columna calculada: el monto total incluyendo un impuesto simulado. Esto es exactamente el tipo de logica de negocio que se agrega en un pipeline real (impuestos, comisiones, descuentos, etc.).

Usamos `withColumn` para crear una columna nueva a partir de una expresion sobre columnas existentes.

In [ ]:
IMPUESTO = 0.13  # Definimos el impuesto como 13%.

df_procesado = df_limpio.withColumn(  # Creamos una nueva columna en el DataFrame.
    "total_con_impuestos",  # Nombre de la nueva columna.
    F.round(F.col("monto") * (1 + IMPUESTO), 2) # Calculamos el monto más el 13% de impuesto. Rounded a dos decimales.
)

df_procesado.select(  # Seleccionamos las columnas que queremos visualizar.
    "monto",  # Mostramos el monto original.
    "total_con_impuestos"  # Mostramos el monto después de aplicar el impuesto.
).show(5)  # Mostramos las primeras 5 filas.


+------+-------------------+
| monto|total_con_impuestos|
+------+-------------------+
| 39.14|              44.23|
|114.84|             129.77|
|247.17|              279.3|
| 49.58|              56.03|
| 69.54|              78.58|
+------+-------------------+
only showing top 5 rows


Equivalente en Pandas:
```
df_procesado = df_limpio.copy()
df_procesado["total_con_impuestos"] = (
    df_procesado["monto"] * (1 + IMPUESTO)
).round(2)

df_procesado[["monto", "total_con_impuestos"]].head(5)
```

También podemos extraer partes de una fecha usando funciones de Spark. Vamos a extraer la hora de cada transaccion, porque la vamos a necesitar para una de las agregaciones de mas adelante.

In [ ]:
df_procesado = df_procesado.withColumn(
    "hora",  # Creamos una nueva columna llamada "hora".
    F.hour(F.col("fecha_hora"))  # Extraemos la hora de la columna "fecha_hora".
)

df_procesado.select(  # Seleccionamos las columnas fecha y hora
    "fecha_hora",
    "hora"
).show(5)  # Mostramos las primeras 5 filas.

+-------------------+----+
|         fecha_hora|hora|
+-------------------+----+
|2024-02-29 22:39:00|  22|
|2024-02-03 17:07:00|  17|
|2024-01-21 05:02:00|   5|
|2024-02-23 03:29:00|   3|
|2024-02-04 18:26:00|  18|
+-------------------+----+
only showing top 5 rows


Equivalente en Pandas:
```
df_procesado["hora"] = df_procesado["fecha_hora"].dt.hour  de "fecha_hora".

df_procesado[["fecha_hora", "hora"]].head(5)  
```

## Filtros avanzados

Podemos combinar varias condiciones en un mismo filtro usando `&` (y) y `|` (o). Por ejemplo, transacciones de alto valor que no se hicieron en tienda fisica (es decir, canales digitales).

In [ ]:
df_alto_valor_digital = df_procesado.filter(  # Filtramos las transacciones que cumplen ambas condiciones.
    (F.col("total_con_impuestos") > 500)  # Nos quedamos con montos superiores a 500.
    & (F.col("canal") != "Tienda Fisica")  # Excluimos las transacciones realizadas en tienda física.
)

df_alto_valor_digital.show(10)  # Mostramos las primeras 10 filas del resultado.

Equivalente en Pandas:
```
df_alto_valor_digital = df_procesado[
    (df_procesado["total_con_impuestos"] > 500)
    & (df_procesado["canal"] != "Tienda Fisica")
]

df_alto_valor_digital.head(10)  
```

## Agregaciones con groupBy

Ahora vamos a calcular las metricas que nos interesan del negocio: el promedio de venta por categoria, y la cantidad de transacciones por hora del dia. Estas son el tipo de metricas que despues vamos a guardar en MySQL.

In [ ]:
# Agrupamos las transacciones por categoría.
resumen_por_categoria = df_procesado.groupBy("categoria") \
    .agg(
        F.avg("total_con_impuestos").alias("promedio_total"),  # Calculamos el promedio del total con impuestos.
        F.count("*").alias("cantidad_transacciones")  # Contamos cuántas transacciones hay en cada categoría.
    ) \
    .orderBy(F.col("promedio_total").desc())  # Ordenamos de mayor a menor promedio.

# alternative sin los \
resumen_por_categoria = (
    df_procesado
    .groupBy("categoria")
    .agg(
        F.avg("total_con_impuestos").alias("promedio_total"),  # Calculamos el promedio del total con impuestos.
        F.count("*").alias("cantidad_transacciones")  # Contamos cuántas transacciones hay en cada categoría.
    )
    .orderBy(F.col("promedio_total").desc())  # Ordenamos de mayor a menor promedio.
)

# Mostramos el resumen por categoría.
resumen_por_categoria.show()

+-----------+------------------+----------------------+
|  categoria|    promedio_total|cantidad_transacciones|
+-----------+------------------+----------------------+
|  Alimentos|171.89365582628207|                  9809|
|       Ropa| 170.2257657114126|                  9945|
|   Deportes|166.07061272504083|                  9776|
| Tecnologia|165.40620332159003|                  9935|
|      Hogar|164.54979538087608|                  9872|
|Electronica| 163.9283085539719|                  9820|
+-----------+------------------+----------------------+



In [ ]:
# Agrupamos las transacciones según la hora.
resumen_por_hora = df_procesado.groupBy("hora") \
    .count() \
    .orderBy("hora")  # Ordenamos las horas de menor a mayor.

# Mostramos las 24 horas del día.
resumen_por_hora.show(24)

+----+-----+
|hora|count|
+----+-----+
|   0| 2439|
|   1| 2545|
|   2| 2498|
|   3| 2471|
|   4| 2484|
|   5| 2357|
|   6| 2454|
|   7| 2423|
|   8| 2491|
|   9| 2474|
|  10| 2497|
|  11| 2374|
|  12| 2322|
|  13| 2453|
|  14| 2446|
|  15| 2442|
|  16| 2578|
|  17| 2443|
|  18| 2468|
|  19| 2526|
|  20| 2490|
|  21| 2530|
|  22| 2483|
|  23| 2469|
+----+-----+



Equivalente en Pandas:
```
resumen_por_hora = (
    df_procesado
    .groupby("hora")
    .size()
    .sort_index()
)

resumen_por_hora.head(24)
```

## Plan de ejecucion y shuffle

Ya hiciste varios `groupBy` en esta sesion. Antes de seguir, vamos a abrir el plan de ejecucion que arma Spark y ver, con nuestros propios ojos, el paso responsable de mover los datos entre particiones quando hacemos ese tipo de operaciones.

Vamos a cubrir:

1. Transformaciones narrow vs wide (la razon real detras del shuffle).
2. Como leer un plan de ejecucion con `.explain()`.
3. Un ejemplo de join, y una sorpresa: Spark a veces evita el shuffle por su cuenta.

## Transformaciones vs Acciones

Ya sabes que en Spark hay dos tipos de operaciones:

- **Transformaciones** (`select`, `filter`, `withColumn`, `groupBy`, etc.): no se ejecutan de inmediato, solo construyen un plan.
- **Acciones** (`show`, `count`, `collect`, etc.): disparan la ejecucion real de ese plan.

Dentro de las transformaciones, hay una distincion importante:

- **Narrow (angostas)**: cada particion de salida depende solo de una particion de entrada. Ejemplos: `select`, `filter`, `withColumn`. Spark puede resolverlas sin mover datos entre particiones.

```text
ENTRADA                  SALIDA
┌─────────────┐          ┌─────────────┐
│ Partición 1 │ ───────► │ Partición 1 │
└─────────────┘          └─────────────┘

┌─────────────┐          ┌─────────────┐
│ Partición 2 │ ───────► │ Partición 2 │
└─────────────┘          └─────────────┘

┌─────────────┐          ┌─────────────┐
│ Partición 3 │ ───────► │ Partición 3 │
└─────────────┘          └─────────────┘

         🚫 No hay movimiento de datos
                 entre particiones
```
<br>

- **Wide (anchas)**: una particion de salida puede depender de datos de varias particiones de entrada. Ejemplos: `groupBy`, `join`, `distinct`, `orderBy`. Estas si necesitan mover datos entre particiones, es decir, provocan un **shuffle**.

```
ENTRADA                         SALIDA

┌─────────────┐
│ Partición 1 │ ──────┐
└─────────────┘       │
                      ├────────► ┌─────────────┐
┌─────────────┐       │          │ Partición 1 │
│ Partición 2 │ ──────┤          └─────────────┘
└─────────────┘       │
                      │
┌─────────────┐       │
│ Partición 3 │ ──────┘
└─────────────┘

            🔄 SHUFFLE
       Los datos se mueven entre
            particiones
```

El shuffle es, en la practica, la operacion mas costosa que existe en Spark: implica escribir datos a disco, moverlos por la red (en un cluster real) y volver a leerlos. Por eso evitar shuffles innecesarios, particionar bien antes de un join, importan tanto.

## Preparando el terreno

Ya tenemos `spark` y `df_spark` cargados de antes en esta misma sesion, los reutilizamos directamente.

## Leyendo un plan de ejecucion con explain()

Todo DataFrame en Spark tiene un metodo `.explain()` que muestra el plan de ejecucion que Spark construyo para esa transformacion, sin necesidad de ejecutarlo. Vamos a empezar con una transformacion narrow: un `filter` simple.

In [ ]:
# Filtramos las filas de transacciones mayor a 1000.
df_spark.filter(F.col("monto") > 1000).explain() # ".explain()" --> Mostramos el plan de ejecución que Spark utilizará para realizar el filtro.


== Physical Plan ==
*(1) Filter (isnotnull(monto#20) AND (monto#20 > 1000.0))
+- FileScan csv [id_transaccion#17,cliente_id#18,categoria#19,monto#20,canal#21,fecha_hora#22] Batched: false, DataFilters: [isnotnull(monto#20), (monto#20 > 1000.0)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/content/data/transacciones_masivas.csv], PartitionFilters: [], PushedFilters: [IsNotNull(monto), GreaterThan(monto,1000.0)], ReadSchema: struct<id_transaccion:int,cliente_id:int,categoria:string,monto:double,canal:string,fecha_hora:ti...




Fijate que el plan solo tiene dos pasos: `Filter` y `FileScan` (la lectura del archivo). No hay ningun paso que mueva datos entre particiones.  
Eso es lo que esperamos de una transformacion narrow: cada particion se filtra de forma independiente, sin necesidad de comunicarse con las demas.

Ahora vamos a ver el plan de una transformacion wide: un `groupBy`.

In [ ]:
# Agrupamos por categoría, contamos los registros de cada grupo y mostramos el execution plan de Spark.
df_spark.groupBy("categoria").count().explain()


== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[categoria#19], functions=[count(1)])
   +- Exchange hashpartitioning(categoria#19, 200), ENSURE_REQUIREMENTS, [plan_id=40]
      +- HashAggregate(keys=[categoria#19], functions=[partial_count(1)])
         +- FileScan csv [categoria#19] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/content/data/transacciones_masivas.csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<categoria:string>




Esta vez el plan es mas largo, y aparece un paso nuevo: `Exchange hashpartitioning(categoria, 200)`. Ese es el shuffle: Spark esta reorganizando los datos para que todas las filas de una misma categoria terminen en la misma particion, y lo hace usando 200 particiones de destino por defecto.

En resumen: cuando en un plan de ejecucion ves la palabra `Exchange`, ahi hay un shuffle.

## Un vistazo mas profundo

Si le pasas `True` a `.explain()`, Spark te muestra no solo el **plan físico final**, sino también los planes anteriores que se generaron durante el proceso.

Primero puedes ver el **plan lógico**, que representa cómo Spark entendió la operación. Después aparece el **plan lógico optimizado**, donde entra en acción el motor **Catalyst**, que mencionamos en la sesión 1. Finalmente, Spark genera el **plan físico**, que es el que se utilizará para ejecutar la operación.

---> Spark pasa por **varias etapas de análisis y optimización** antes de llegar al plan físico final.


In [ ]:
df_spark.filter(F.col("monto") > 1000).explain(True)


== Parsed Logical Plan ==
'Filter '`>`('monto, 1000)
+- Relation [id_transaccion#17,cliente_id#18,categoria#19,monto#20,canal#21,fecha_hora#22] csv

== Analyzed Logical Plan ==
id_transaccion: int, cliente_id: int, categoria: string, monto: double, canal: string, fecha_hora: timestamp
Filter (monto#20 > cast(1000 as double))
+- Relation [id_transaccion#17,cliente_id#18,categoria#19,monto#20,canal#21,fecha_hora#22] csv

== Optimized Logical Plan ==
Filter (isnotnull(monto#20) AND (monto#20 > 1000.0))
+- Relation [id_transaccion#17,cliente_id#18,categoria#19,monto#20,canal#21,fecha_hora#22] csv

== Physical Plan ==
*(1) Filter (isnotnull(monto#20) AND (monto#20 > 1000.0))
+- FileScan csv [id_transaccion#17,cliente_id#18,categoria#19,monto#20,canal#21,fecha_hora#22] Batched: false, DataFilters: [isnotnull(monto#20), (monto#20 > 1000.0)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/content/data/transacciones_masivas.csv], PartitionFilters: [], PushedFilters: [IsNotNull(monto), 

## Joins: otra transformacion wide

Un `join` tambien es una transformacion wide en teoria, porque Spark necesita juntar filas que pueden estar en particiones distintas. Vamos a probarlo con una tabla pequena de apoyo, que describe cada canal de venta.

In [ ]:
canal_info = spark.createDataFrame( # creamos un pequeno dataframe que indica nombre corto de las categorias
    [
        ("App", "Aplicacion movil"),
        ("Web", "Sitio web"),
        ("Tienda Fisica", "Punto de venta"),
    ],
    ["canal", "descripcion"],
)

canal_info.show()


+-------------+----------------+
|        canal|     descripcion|
+-------------+----------------+
|          App|Aplicacion movil|
|          Web|       Sitio web|
|Tienda Fisica|  Punto de venta|
+-------------+----------------+



In [ ]:
df_join = df_spark.join(canal_info, on="canal")  # Hacemos un join entre ambos DataFrames usando la columna canal como clave.
df_join.show(10)

+-----+--------------+----------+-----------+------+-------------------+----------------+
|canal|id_transaccion|cliente_id|  categoria| monto|         fecha_hora|     descripcion|
+-----+--------------+----------+-----------+------+-------------------+----------------+
|  App|         46801|      3692| tecnologia| 52.45|2024-01-30 18:31:00|Aplicacion movil|
|  App|         16651|      4570|  alimentos| 50.41|2024-02-28 06:51:00|Aplicacion movil|
|  App|          7318|      1262|      Hogar|205.02|2024-02-05 03:35:00|Aplicacion movil|
|  App|         17933|      2069|       ROPA| 26.16|2024-02-23 12:26:00|Aplicacion movil|
|  App|         33150|      4363|Electronica|  91.5|2024-01-10 09:01:00|Aplicacion movil|
|  App|         34404|      4520| TECNOLOGIA| 364.7|2024-01-28 09:52:00|Aplicacion movil|
|  App|         45562|      4621|  alimentos|148.23|2024-01-22 01:42:00|Aplicacion movil|
|  App|         10477|      1838|       Ropa|140.06|2024-02-25 15:06:00|Aplicacion movil|
|  App|   

In [ ]:
df_join.explain()  # Mostramos el execution plan que Spark utilizará para realizar el join.

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [canal#21, id_transaccion#17, cliente_id#18, categoria#19, monto#20, fecha_hora#22, descripcion#35]
   +- BroadcastHashJoin [canal#21], [canal#34], Inner, BuildLeft, false
      :- BroadcastExchange HashedRelationBroadcastMode(List(input[4, string, false]),false), [plan_id=297]
      :  +- Filter isnotnull(canal#21)
      :     +- FileScan csv [id_transaccion#17,cliente_id#18,categoria#19,monto#20,canal#21,fecha_hora#22] Batched: false, DataFilters: [isnotnull(canal#21)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/content/data/transacciones_masivas.csv], PartitionFilters: [], PushedFilters: [IsNotNull(canal)], ReadSchema: struct<id_transaccion:int,cliente_id:int,categoria:string,monto:double,canal:string,fecha_hora:ti...
      +- Filter isnotnull(canal#34)
         +- Scan ExistingRDD[canal#34,descripcion#35]




Aca no se ve ningun `Exchange`. En su lugar aparece un `BroadcastHashJoin`.  

Spark se dio cuenta de que `canal_info` es una tabla diminuta (tres filas), y en lugar de hacer un shuffle costoso, le mando una copia completa de esa tabla a cada particion. A esto se le llama **broadcast join**, y es una optimizacion automatica que Spark aplica cuando una de las dos tablas de un join es lo suficientemente pequena (por defecto, menos de 10 MB).

En otras palabras: aunque el join es conceptualmente una operacion wide, el optimizador de Spark evito el shuffle por su cuenta porque encontro una forma mas barata de resolverlo. Vamos a forzar el comportamiento clasico (sin esta optimizacion) para comparar.

In [ ]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")  # Desactivamos el límite que permite a Spark usar Broadcast Join automáticamente.

df_join_forzado = df_spark.join(canal_info, on="canal")  # Hacemos el join entre ambos DataFrames usando canal como clave.

df_join_forzado.explain()  # Mostramos el execution plan para verificar qué estrategia de join está usando Spark.

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [canal#21, id_transaccion#17, cliente_id#18, categoria#19, monto#20, fecha_hora#22, descripcion#35]
   +- SortMergeJoin [canal#21], [canal#34], Inner
      :- Sort [canal#21 ASC NULLS FIRST], false, 0
      :  +- Exchange hashpartitioning(canal#21, 200), ENSURE_REQUIREMENTS, [plan_id=327]
      :     +- Filter isnotnull(canal#21)
      :        +- FileScan csv [id_transaccion#17,cliente_id#18,categoria#19,monto#20,canal#21,fecha_hora#22] Batched: false, DataFilters: [isnotnull(canal#21)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/content/data/transacciones_masivas.csv], PartitionFilters: [], PushedFilters: [IsNotNull(canal)], ReadSchema: struct<id_transaccion:int,cliente_id:int,categoria:string,monto:double,canal:string,fecha_hora:ti...
      +- Sort [canal#34 ASC NULLS FIRST], false, 0
         +- Exchange hashpartitioning(canal#34, 200), ENSURE_REQUIREMENTS, [plan_id=328]
            +- Filter isnotnull(

Ahora si aparece un `SortMergeJoin`, junto con dos pasos de `Exchange`, uno por cada lado del join. Este es el comportamiento clasico que uno esperaria de una transformacion wide: los datos de ambos DataFrames se reparticionan (shuffle) antes de poder juntarse.

La leccion aqui no es que el broadcast join sea siempre mejor, sino que el plan de ejecucion te permite ver exactamente que decision tomo Spark, en lugar de asumirlo.

### En resumen (plan de ejecucion)

Ya tienes las piezas para leer un plan de ejecucion de Spark:

- `Filter`, `Project`, `FileScan`: pasos normales, sin shuffle.
- `Exchange`: aca es donde ocurre un shuffle, casi siempre causado por un `groupBy`, un `join` sin broadcast, o un `orderBy`.
- `BroadcastHashJoin` vs `SortMergeJoin`: dos formas distintas de resolver un join, con costos muy diferentes.

La proxima vez que una operacion en Spark tarde mas de lo esperado, `.explain()` es la primera herramienta a la que puedes recurrir para entender que esta pasando por debajo.

## Errores comunes al trabajar con PySpark

Antes de cerrar, vale la pena mencionar tres errores frecuentes al trabajar con datasets grandes en Spark:

- **Error de memoria (OOM)**: convertir un DataFrame de Spark gigante a Pandas con `.toPandas()` sin filtrar antes. Esto intenta traer todos los datos distribuidos a la memoria de una sola maquina.
- **No definir el schema**: dejar `inferSchema=True` en archivos muy grandes puede ser lento, porque obliga a Spark a leer el archivo completo una vez extra solo para deducir los tipos.
- **Shuffle excesivo**: hacer demasiados `join` o `groupBy` entre tablas muy grandes sin particionar antes, lo que ralentiza el movimiento de datos entre los nodos del cluster.

La siguiente celda no se debe ejecutar, es solo un ejemplo de lo que hay que evitar en un dataset realmente masivo.

In [ ]:
# Ejemplo de lo que NO se debe hacer con un dataset realmente masivo:
# df_pandas = df_procesado.toPandas()
# Esto puede saturar la memoria si el DataFrame tiene millones de filas.


## Cierre de la sesion

Ya tienes un DataFrame limpio, con columnas calculadas y con las metricas agregadas que le interesan al negocio: `resumen_por_categoria` y `resumen_por_hora`.

En la proxima sesion vamos a:

- Conectar Python a una base de datos MySQL.
- Crear una tabla para recibir estos resultados.
- Insertar los datos usando sentencias preparadas.
- Ejecutar un `SELECT` final para confirmar que todo se guardo correctamente.

Esa ultima sesion cierra el pipeline completo: extraccion, procesamiento y persistencia.